In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
!pip install pylint langchain langchain-text-splitters langchain-community llama_index langchain-google-genai google-generativeai python-dotenv

In [2]:
import tempfile

from dotenv import load_dotenv
import os
load_dotenv()

from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_parse import LlamaParse
from llama_index.core import  VectorStoreIndex, SummaryIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.core.selectors.utils import get_selector_from_llm

api_key = os.getenv("GOOGLE_API_KEY")
llama_parser_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

Settings.llm = GoogleGenAI(model="gemini-2.5-flash-lite", api_key=api_key)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=api_key)


parser = LlamaParse(
    api_key= llama_parser_api_key,
    result_type="markdown",
)
documents = parser.load_data("../data/clean_code_full.pdf")

vectorIndex=VectorStoreIndex.from_documents(documents)
summaryIndex=SummaryIndex.from_documents(documents)

vectorEngine=vectorIndex.as_query_engine(similarity_top_k=5)
summaryEngine=summaryIndex.as_query_engine(
    responses_model='tree_summarize',
    use_async=True
)

vectorTool = QueryEngineTool.from_defaults(
    query_engine=vectorEngine,
    description=(
        "Use this for questions that need specific facts, quotes, definitions, "
        "or details from the document."
    ),
)
summaryTool = QueryEngineTool.from_defaults(
    query_engine=summaryEngine,
    description=(
        "Use this for summarization questions, high-level overviews, or "
        "when the user asks to summarize a chapter/section/document."
    ),
)

tools = [vectorTool, summaryTool]
selector=get_selector_from_llm(Settings.llm)

routerEngine=RouterQueryEngine(
    selector=selector,
    query_engine_tools=tools,
    verbose=True
)

# librarian_tool = QueryEngineTool.from_defaults(
#     query_engine=routerEngine,
#     name="clean_code_librarian",
#     description=(
#         "Use this tool to retrieve principles, quotes, and explanations "
#         "from Robert C. Martin's book 'Clean Code'. "
#         "Use it when you need justification or guidance about why a piece "
#         "of code is bad or violates clean coding practices."
#     )
# )






Started parsing the file under job_id 39ef60d2-6f88-4103-a61f-2ee8c95a245e


In [ ]:
response=routerEngine.query(input("enter your query:"))
print(response)

In [3]:
import json
import os
import subprocess
import tempfile
from langchain_core.tools import tool


@tool
def pylint_linter(code: str) -> dict:
    """
    Runs pylint on Python code and returns only magic number comparison issues.
    """
    file_path = None

    try:
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".py",
            mode="w",
            encoding="utf-8"
        ) as f:
            f.write(code)
            file_path = f.name

        result = subprocess.run(
            [
                "pylint",
                file_path,
                "--output-format=json",
                "--load-plugins=pylint.extensions.magic_value",
                "--disable=all",
                "--enable=magic-value-comparison",
            ],
            capture_output=True,
            text=True,
            encoding="utf-8"
        )

        stdout = result.stdout.strip()
        issues = json.loads(stdout) if stdout else []

        return {
            "line_count": len(code.splitlines()),
            "magic_number_issues": issues,
        }

    except FileNotFoundError:
        return {"error": "pylint is not installed or not found in PATH"}
    except json.JSONDecodeError as e:
        return {"error": f"Invalid JSON from pylint: {e}"}
    except Exception as e:
        return {"error": str(e)}
    finally:
        if file_path and os.path.exists(file_path):
            os.remove(file_path)

In [6]:
code = """
def add(a,b):
    if x == 10:
        return 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    result = a + b + 10
    return result
add(5, 10)
"""

print(pylint_linter.invoke(code))

{'line_count': 59, 'magic_number_issues': [{'type': 'refactor', 'module': 'tmpny74afq_', 'obj': 'add', 'line': 3, 'column': 7, 'endLine': 3, 'endColumn': 14, 'path': 'C:\\Users\\pspra\\AppData\\Local\\Temp\\tmpny74afq_.py', 'symbol': 'magic-value-comparison', 'message': "Consider using a named constant or an enum instead of '10'.", 'message-id': 'R2004'}]}


In [10]:
load_dotenv()
SYSTEM_PROMPT = """
You are a Code Auditor that evaluates Python code quality.

You have access to two tools:

1. code_linter:
Use this tool to analyze the given code snippet and it should focus on two things
    1. number of lines in the function
    2. the presence of magic numbers


2. clean_code_librarian:
Use this tool to retrieve principles or quotes from the book 'Clean Code'
by Robert C. Martin that explain why certain coding practices are bad.

Your workflow:

Step 1: Use the code_linter tool to analyze the code and identify measurable issues like heavy function defination or magical numbers presence .
Step 2: Based on the issues found, use the clean_code_librarian tool to retrieve
relevant Clean Code principles.
Step 3: Produce a final explanation describing why the code is unclean,
using both the metrics from the linter and the Clean Code principle.

Your response should clearly explain:
- What the problem in the code is
- The metric detected by the linter
- The Clean Code principle that justifies the issue.

Be concise but clear in your explanation.
"""

from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
from langchain_core.tools import tool

@tool
def clean_code_librarian(query: str) -> str:
    """
    Retrieve Clean Code principles from the Clean Code book
    to justify why a code practice is unclean.
    """
    response = routerEngine.query(query)
    return str(response)

tools = [pylint_linter, clean_code_librarian]



agent = create_agent(
    llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)

response = agent.invoke({"messages": code})

print(response)

Selecting query engine 0: The user is asking for a specific fact ('magic numbers') which requires retrieving details from the document..


Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash-lite\nPlease retry in 27.265566285s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash-lite', 'location

In [8]:
import json

print(json.dumps(response))

TypeError: Object of type HumanMessage is not JSON serializable